# Suite SVD — The SystemVerilog Preprocessor dialect

`Expressions.SystemVerilog.Preprocessor` models Verilog's and SystemVerilog's compiler directives and macro usages
(IEEE 1800 clause 22), and the comments kept among them, as mbse-programs has them: a macro usage where an expression
stands, and the directives and comments where a constraint block's items, or `randomize() with`'s, are listed. Its
kinds are validated, bind nothing but a conditional directive's macros' names within its condition, cover every edition
of IEEE 1364 and IEEE 1800, print as mbse-programs prints them, and are refused by evaluation by name.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import { Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const [SV, SVV, SVText] = [SystemVerilog.Expressions, SystemVerilog.Evaluators, SystemVerilog.Text];
const [PP, PPText, SVC] = [SVPreprocessor.Expressions, SVPreprocessor.Text, SVConstraints.Expressions];

const store: any = new Builders.Store(); // the SystemVerilog family's six dialects, the Preprocessor's install before the others'
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, SVPreprocessor.install, SystemVerilog.install,
  SVTypes.install, SVPatterns.install, SVAssertions.install, SVConstraints.install]) {
  store.transaction(install);
}
for (const define of [SystemVerilog.define, SVPreprocessor.define]) store.transaction(define); // SystemVerilog's types, and the Preprocessor's editions

/** The term value `spec` writes, the root of `dialect`'s holder written in a transaction. */
function term(spec: unknown, dialect: F.Dialect = PP.DIALECT): any {
  return F.term(store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec)))));
}

const validate = (spec: unknown, bound: string[] = [], dialect: F.Dialect = PP.DIALECT): string[] =>
  dialect.validate(term(spec, dialect), bound);
const text = (spec: unknown, dialect: F.Dialect = PP.DIALECT): string => SVText.ToText(term(spec, dialect));
const union = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name); // a union's branches
const spelled = (word: string) => (i: any) => i.spelling(word); // an identifier, of its spelling
const name = (word: string) => (u: any) => u.nameexpression((n: any) => n.name((i: any) => i.identifier(spelled(word))));
const parenthesized = (inner: unknown) => (u: any) => u.parenthesizedexpression((p: any) => p.expression(inner));
const comment = (words: string, block = false) => (u: any) => u.comment((c: any) => c.text(words).block(block));
const [x, y] = [name("x"), name("y")];
const condition = parenthesized(SV.binary("&&", name("A"), SV.unary("!", name("B")))); // (A && !B)
const ifdef = (u: any) => u.ifdefdirective((d: any) => d.condition(condition).items([SV.binary(">", x, 0n), comment(" low")])
  .branches([(b: any) => b.name(spelled("C")).items([SV.binary("<", x, 9n)])]).has_else(true)); // an empty `else
const kinds = new Map((store.registered(PP.DIALECT.union).branches as any[]).map((b) => [b.name, b.schema] as [string, any]));
const props = (schema: any): string[] => [...schema.properties.keys()];
const registered = (helper: string): any => store.registered(`Expressions.SystemVerilog.Preprocessor.${helper}`);

<a id="SVD-01"></a>
## SVD-01 · Its union and holder: its kinds and invalid syntax, the directives and comments a constraint block's items too, a holder refining the framework's, linked by HasRoot, round-tripped through JSON

In [ ]:
{
  // Its Term: its kinds, mbse-programs', then invalid syntax; Constraints' Term lists the directives and comment beside
  // the macro usage it lists as Expressions' Term does, and Expressions' branches.
  assert(same(PP.TAGS, ["macrousage", "includedirective", "definedirective", "undefdirective", "timescaledirective",
    "defaultnettypedirective", "ifdefdirective", "disabledtext", "otherdirective", "comment"]));
  assert(same(union(PP.DIALECT.union), [...PP.TAGS, "error", "missing"]) && same([...PP.DIALECT.kinds().keys()], [...PP.TAGS, "error", "missing"]));
  assert(same(PP.DIRECTIVES, PP.TAGS.slice(1, -1)) && same(PP.HELPERS, ["ElsifDirective"]));
  assert(same(union(SVC.DIALECT.union).slice(0, 10), PP.TAGS) && SV.TAGS.every((tag) => union(SVC.DIALECT.union).includes(tag)));
  assert(union(SV.DIALECT.union).includes("macrousage") && !union(SV.DIALECT.union).includes("comment")); // an expression holds none
  const held = new Map((store.registered(SVC.DIALECT.union).branches as any[]).map((b) => [b.name, b.schema] as [string, any]));
  assert(PP.TAGS.every((tag) => kinds.get(tag) === held.get(tag))); // one schema, a branch of each union that holds it
  assert(PP.DIALECT.name() === "SystemVerilog Preprocessor" && PP.DIALECT.tag === "systemverilog_preprocessor");
  assert(union("Expressions.Framework.Term").includes("systemverilog_preprocessor"));
  // Its holder: a reference object refining the framework's Expression, its root narrowed to its Term.
  const Holder: any = store.registered("Expressions.SystemVerilog.Preprocessor.Expression");
  assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(PP.DIALECT.union) && Holder.validate().length === 0);
  const [holder, linked] = store.transaction(() => {
    const holder = store.Expressions.SystemVerilog.Preprocessor.Expression((e: any) => e.root(PP.DIALECT.spec(ifdef)));
    const linked = store.Schemas.OfExpression((s: any) => s.name("svd.ifdef").expression(
      (a: any) => a.insert((e: any) => e.expression(holder))));
    return [holder, linked];
  });
  assert(Validators.Validate(store)(Holder, holder).length === 0 && PP.DIALECT.validate(holder, ["x"]).length === 0);
  assert(F.dialectOf(F.term(holder)) === PP.DIALECT.union && F.kindOf(F.term(holder))[0] === "ifdefdirective");
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)) as any).root, holder.root));
  assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, holder)), [true]));
  raises(Error, () => store.transaction(() => // its root holds its own terms
    store.Expressions.SystemVerilog.Preprocessor.Expression((e: any) => e.root(PP.DIALECT.spec(SV.binary("+", x, 1n)))),
  ), "'binaryexpression' is not a member of a union value");
}

<a id="SVD-02"></a>
## SVD-02 · Its kinds and helpers: mbse-programs' properties in its order, a directive's branches holding Constraints' terms, its condition Expressions'

In [ ]:
{
  assert(same(props(kinds.get("macrousage")), ["name", "arguments"]));
  assert(same(props(kinds.get("includedirective")), ["path", "system"]));
  assert(same(props(kinds.get("definedirective")), ["name", "function_like", "parameters", "body"]));
  assert(same(props(kinds.get("undefdirective")), ["name"]) && same(props(kinds.get("timescaledirective")), ["unit", "precision"]));
  assert(same(props(kinds.get("defaultnettypedirective")), ["net_type"]));
  assert(same(props(kinds.get("ifdefdirective")), ["negated", "name", "condition", "items", "branches", "alternative", "has_else"]));
  assert(same(props(registered("ElsifDirective")), ["name", "condition", "items"]));
  assert(same(props(kinds.get("disabledtext")), ["text"]) && same(props(kinds.get("otherdirective")), ["text"]));
  assert(same(props(kinds.get("comment")), ["block", "text", "trailing"]));
  const IFDEF = kinds.get("ifdefdirective");
  assert(IFDEF.properties.get("condition").schema === store.registered(SV.DIALECT.union));
  assert(IFDEF.properties.get("items").schema.element === store.registered(SVC.DIALECT.union));
  assert(IFDEF.properties.get("alternative").schema.element === store.registered(SVC.DIALECT.union));
  assert(IFDEF.properties.get("name").schema === store.registered("Expressions.SystemVerilog.Expressions.Identifier"));
  assert(equal(F._parts(IFDEF), [["condition"], ["items", "alternative"], ["negated", "name", "branches", "has_else"]]));
  assert(PP.HELPERS.every((h) => !registered(h).ref && !kinds.has(registered(h).name.toLowerCase().split(".").at(-1))));
  // Its terms' nodes: the condition a slot, its items and its `else`'s two containers of terms, read apart (`lists`),
  // each branch a helper holding its own. Its form gives each container's size, so that make puts each term back.
  const node = F.node(...F.kindOf(term(ifdef)));
  const branches = node.helpers.get("branches") as F.Node[];
  assert(node.many.length === 2 && same([...node.lists.keys()], ["items", "alternative"]) && same(node.lists.get("items")!, node.many)
    && node.lists.get("alternative")!.length === 0 && node.attributes.get("has_else") === true);
  assert(same(branches.map((b) => b.kind), ["elsifdirective"]) && same(branches[0]!.lists.get("items")!, branches[0]!.many)
    && equal((branches[0]!.helpers.get("name") as F.Node).attributes, map({ spelling: "C" })));
  assert(!F.node(...F.kindOf(term((u: any) => u.ifdefdirective((d: any) => d.name(spelled("A")))))).attributes.has("has_else"));
  const form = F.form(term(ifdef));
  const sizes = ["items", "alternative"].map((name) => form.attributes.get(name) as F.Part);
  assert(same(sizes.map((p) => [p.kind, p.attributes.size, p.size].join(" ")), ["items 0 2", "alternative 0 0"]));
  assert(F.same(term(PP.DIALECT.make(form)), term(ifdef)));
  // Given no sizes, a container holds none, but the last, which holds the rest; sizes that do not add up are refused.
  const bare = new F.Form(form.kind, new Map([...form.attributes].filter(([k]) => !["items", "alternative"].includes(k))), form.arguments);
  const lists = F.node(...F.kindOf(term(PP.DIALECT.make(bare)))).lists;
  assert(lists.get("items")!.length === 0 && same(lists.get("alternative")!.map((t) => F.form(t).kind), node.many.map((t) => F.form(t).kind)));
  raises(ValueError, () => term(PP.DIALECT.make(new F.Form(form.kind, new Map([...form.attributes, ["items", new F.Part("items", new Map(), 5)]]),
    form.arguments))), "an ifdefdirective's items, alternative hold 5 arguments, got 2");
}

<a id="SVD-03"></a>
## SVD-03 · Its checks: what each kind requires, a default net type, an identifier's spelling, and a conditional directive's name or condition, with mbse-programs' messages

In [ ]:
{
  const undef = (word: string) => (u: any) => u.undefdirective((d: any) => d.name(spelled(word)));
  const nettype = (kind: string) => (u: any) => u.defaultnettypedirective((d: any) => d.net_type(kind));
  const ifdefOf = (build: (d: any) => any) => (u: any) => u.ifdefdirective(build);
  const attributed = [(a: any) => a.specs([(s: any) => s.name(spelled("k"))])];
  const only = "an IfdefDirective's condition has macros' names, !, &&, ||, -> and <-> only";
  const cases: [unknown, string[]][] = [
    [(u: any) => u.macrousage((m: any) => m.name(spelled("WIDTH")).arguments("a, b")), []],
    [(u: any) => u.macrousage((m: any) => m.arguments("a")), ["a macrousage needs a name"]],
    [(u: any) => u.includedirective((d: any) => d.path("defs.svh").system(true)), []],
    [(u: any) => u.includedirective((d: any) => d.path("")), ["an includedirective needs a path"]],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("F")).function_like(true).parameters([spelled("a"), spelled("2b")])),
      ["'2b' is not an identifier"]],
    [(u: any) => u.definedirective((d: any) => d.body("1")), ["a definedirective needs a name"]],
    [undef("WIDTH"), []], [undef("9"), ["'9' is not an identifier"]],
    [(u: any) => u.undefdirective((d: any) => d), ["an undefdirective needs a name"]],
    [(u: any) => u.timescaledirective((d: any) => d.unit("1ns")), ["a timescaledirective needs a precision"]],
    [nettype("none"), []], [nettype("uwire"), []], [nettype("wirex"), ["DefaultNettypeDirective.net_type cannot be 'wirex'"]],
    [(u: any) => u.defaultnettypedirective((d: any) => d), ["a defaultnettypedirective needs a net_type"]],
    [(u: any) => u.disabledtext((d: any) => d.text("")), ["a disabledtext needs a text"]],
    [(u: any) => u.otherdirective((d: any) => d), ["an otherdirective needs a text"]],
    [(u: any) => u.otherdirective((d: any) => d.text("`resetall")), []],
    [comment(""), []], [(u: any) => u.comment((c: any) => c), []], // `//`: an empty comment is one
    [(u: any) => u.comment((c: any) => c.block(1n)), ["a comment's block must be a bool, got int"]],
    // a conditional directive: a name or a condition, in parentheses, of macros' names, !, &&, ||, -> and <->
    [ifdefOf((d) => d.name(spelled("A")).negated(true)), []],
    [ifdefOf((d) => d.condition(parenthesized(SV.binary("->", name("A"), SV.binary("<->", name("B"), parenthesized(name("C"))))))), []],
    [ifdefOf((d) => d), ["an IfdefDirective has a name or a condition"]],
    [ifdefOf((d) => d.name(spelled("A")).condition(condition)), ["an IfdefDirective has a name or a condition"]],
    [ifdefOf((d) => d.condition(name("A"))), ["an IfdefDirective's condition is in parentheses"]],
    [ifdefOf((d) => d.condition(parenthesized(SV.binary("+", name("A"), name("B"))))), [only]],
    [ifdefOf((d) => d.condition(parenthesized(SV.unary("~", name("A"))))), [only]],
    [ifdefOf((d) => d.condition(parenthesized((u: any) => u.unaryexpression((r: any) => r.operator("!").operand(name("A"))
      .attributes(attributed))))), [only]],
    [ifdefOf((d) => d.condition(parenthesized((u: any) => u.binaryexpression((r: any) => r.left(name("A")).operator("&&")
      .attributes(attributed).right(name("B")))))), [only]],
    [ifdefOf((d) => d.condition(parenthesized((u: any) => u.nameexpression((n: any) => n.name((s: any) => s.scopedname(
      (q: any) => q.scope((i: any) => i.identifier(spelled("p"))).name((i: any) => i.identifier(spelled("A"))))))))), [only]],
    [ifdefOf((d) => d.name(spelled("A")).branches([(b: any) => b.items([])])), ["an ElsifDirective has a name or a condition"]],
    [ifdefOf((d) => d.name(spelled("A")).branches([(b: any) => b.condition(name("B"))])), ["an ElsifDirective's condition is in parentheses"]],
    [ifdefOf((d) => d.name(spelled("1A"))), ["'1A' is not an identifier"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec), problems), `${problems} != ${validate(spec)}`);
}

<a id="SVD-04"></a>
## SVD-04 · What it binds: a conditional directive's macros' names, within its condition alone; its branches' items are read in their scope

In [ ]:
// A condition's names name macros, defined or not, never values: each is bound within its condition, an elsif's too,
// and not in the branches' items, whose names must be bound as anywhere else.
assert(same(validate(ifdef), ["items 0: left: nameexpression 'x' is not bound", "branches 0 argument 0: left: nameexpression 'x' is not bound"]));
assert(validate(ifdef, ["x"]).length === 0 && same([...SY.free(term(ifdef), PP.DIALECT)], ["x"]));
{
  const usesMacro = (u: any) => u.ifdefdirective((d: any) => d.name(spelled("A")).items([name("A")])); // `ifdef A A `endif
  assert(same(validate(usesMacro), ["items 0: nameexpression 'A' is not bound"])); // a name in its items is a value's
  const elsif = (u: any) => u.ifdefdirective((d: any) => d.name(spelled("A")).branches([
    (b: any) => b.condition(parenthesized(SV.binary("||", name("B"), name("C")))).items([y])]));
  assert(same(validate(elsif), ["branches 0 argument 0: nameexpression 'y' is not bound"]) && same([...SY.free(term(elsif), PP.DIALECT)], ["y"]));
  assert(!PP.TAGS.some((tag) => F.isBinder(kinds.get(tag)))); // none refines Let
}

<a id="SVD-05"></a>
## SVD-05 · Its kinds among the family's: directives and comments among a randomize() with's constraints and a constraint block's items, validated and printed from Expressions', Constraints' and its own holders

In [ ]:
{
  // `randomize() with` lays out its constraints on one line, as mbse-programs does: a line comment ends its line, a
  // directive takes lines of its own, a space between the others but not at a line's end or start.
  const randomize = (...items: unknown[]) => (u: any) => u.randomizewithexpression(
    (r: any) => r.call(SV.method(x, "randomize")).items(items));
  const mixed = randomize(SV.binary(">", x, 0n), comment(" c"), comment(" b ", true), ifdef, SV.binary("<", x, 5n),
    (u: any) => u.macrousage((m: any) => m.name(spelled("MORE"))));
  assert(validate(mixed, ["x"], SV.DIALECT).length === 0);
  assert(text(mixed, SV.DIALECT) === "x.randomize() with { x > 0; // c\n/* b */\n`ifdef (A && !B)\nx > 0;\n// low\n"
    + "`elsif C\nx < 9;\n`else\n`endif\nx < 5; `MORE }");
  assert(text(randomize(comment(" only")), SV.DIALECT) === "x.randomize() with { // only\n}");
  assert(text(randomize((u: any) => u.otherdirective((d: any) => d.text("`resetall"))), SV.DIALECT) === "x.randomize() with {\n`resetall\n}");
  assert(text(randomize(SV.binary(">", x, 0n), SV.binary("<", x, 5n)), SV.DIALECT) === "x.randomize() with { x > 0; x < 5; }");
  // The same terms, rooted in Constraints' holder and its own; a directive's branches hold Constraints' terms, invalid
  // syntax among them, and a nested directive.
  const nested = (u: any) => u.ifdefdirective((d: any) => d.negated(true).name(spelled("SIM")).items([
    (i: any) => i.timescaledirective((t: any) => t.unit("1ns").precision("1ps")), ifdef,
    (i: any) => i.error((e: any) => e.text("x >"))]));
  for (const dialect of [SVC.DIALECT, PP.DIALECT]) {
    assert(same(validate(nested, ["x"], dialect), ["items 2: invalid syntax 'x >'"]));
    assert(text(nested, dialect) === "`ifndef SIM\n`timescale 1ns / 1ps\n`ifdef (A && !B)\nx > 0;\n// low\n`elsif C\n"
      + "x < 9;\n`else\n`endif\nx >\n`endif");
  }
  assert(PPText.ToText === SVText.ToText && validate(comment(" c"), [], SVC.DIALECT).length === 0);
}

<a id="SVD-06"></a>
## SVD-06 · Invalid syntax: error and missing among its terms, validated and printed as written, an error with no text refused by name

In [ ]:
{
  const error = (u: any) => u.error((e: any) => e.text("`define"));
  const missing = (u: any) => u.missing((m: any) => m.kind("macrousage"));
  assert(same(validate(error), ["invalid syntax '`define'"]) && same(validate(missing), ["missing macrousage"]));
  assert(text(error) === "`define" && text(missing) === "");
  raises(ValueError, () => text((u: any) => u.error((e: any) => e)), "SystemVerilog Preprocessor error has no text to print");
}

<a id="SVD-07"></a>
## SVD-07 · Its editions: Verilog's and SystemVerilog's eight, each by its designation and common name, each with every one of its kinds

In [ ]:
{
  const own = store.Instances.Expressions.SystemVerilog.Preprocessor;
  assert(same([...FS.standards(own).keys()], SV.EDITIONS.map(([designation]) => designation)));
  assert(equal([...FS.standards(own).values()].map((s) => FS.tags(s)), SV.EDITIONS.map(([designation, , , common]) => [designation, common])));
  assert(equal(FS.properties(FS.find(own, "Verilog-1995")!), map({ bodies: ["org.ieee"], number: "1364", revision: ["year", 1995n],
    names: ["Verilog-1995"] })));
  assert(FS.find(own, "IEEE Std 1800-2023") === FS.find(own, "SystemVerilog-2023") && PP.SINCE.size === 0);
  const every = PP.TAGS.map((tag) => kinds.get(tag)); // each of its kinds in every edition; helpers and invalid syntax in none
  assert([...FS.standards(own).values()].every((s) => same(FS.kinds(s), every)));
  assert(same([...FS.of(own, kinds.get("comment")).keys()], [...FS.standards(own).keys()]) && FS.of(own, registered("ElsifDirective")).size === 0);
  store.transaction(SVPreprocessor.define); // run again, it writes nothing
  assert(FS.standards(own).size === 8);
}

<a id="SVD-08"></a>
## SVD-08 · Printing: each kind as mbse-programs' printer writes it, a directive at the start of its line, from any holder of the family

In [ ]:
{
  const cases: [unknown, string][] = [
    [(u: any) => u.macrousage((m: any) => m.name(spelled("WIDTH"))), "`WIDTH"],
    [(u: any) => u.macrousage((m: any) => m.name(spelled("MAX")).arguments("a, b")), "`MAX(a, b)"],
    [(u: any) => u.includedirective((d: any) => d.path("defs.svh")), '`include "defs.svh"'],
    [(u: any) => u.includedirective((d: any) => d.path("uvm.svh").system(true)), "`include <uvm.svh>"],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("W"))), "`define W"],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("W")).body("8")), "`define W 8"],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("MAX")).function_like(true).parameters([spelled("a"), spelled("b")])
      .body("((a) > (b) ? (a) : (b))")), "`define MAX(a, b) ((a) > (b) ? (a) : (b))"],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("F")).function_like(true).body("1 \\\n+ 2")), "`define F() 1 \\\n+ 2"],
    [(u: any) => u.undefdirective((d: any) => d.name(spelled("W"))), "`undef W"],
    [(u: any) => u.timescaledirective((d: any) => d.unit("1ns").precision("10ps")), "`timescale 1ns / 10ps"],
    [(u: any) => u.defaultnettypedirective((d: any) => d.net_type("none")), "`default_nettype none"],
    [(u: any) => u.ifdefdirective((d: any) => d.name(spelled("A"))), "`ifdef A\n`endif"],
    [(u: any) => u.ifdefdirective((d: any) => d.negated(true).name(spelled("A")).items([(i: any) => i.disabledtext((t: any) => t.text("x < 1;\nx > 2;"))])
      .alternative([SV.binary("==", x, 1n)]).has_else(true)), "`ifndef A\nx < 1;\nx > 2;\n`else\nx == 1;\n`endif"],
    [ifdef, "`ifdef (A && !B)\nx > 0;\n// low\n`elsif C\nx < 9;\n`else\n`endif"],
    [(u: any) => u.disabledtext((t: any) => t.text("a;\nb;")), "a;\nb;"],
    [(u: any) => u.otherdirective((d: any) => d.text("`celldefine")), "`celldefine"],
    [comment(" note"), "// note"], [comment(" note\n * more ", true), "/* note\n * more */"],
    [(u: any) => u.comment((c: any) => c.trailing(true)), "//"],
  ];
  for (const [spec, source] of cases) assert(text(spec) === source, `${source} != ${text(spec)}`);
  assert(text((u: any) => u.macrousage((m: any) => m.name(spelled("W"))), SV.DIALECT) === "`W"); // from Expressions' holder
  assert(text(SV.binary("+", (u: any) => u.macrousage((m: any) => m.name(spelled("W"))), 1n), SV.DIALECT) === "`W + 1");
  for (const spec of [(u: any) => u.ifdefdirective((d: any) => d),
    (u: any) => u.ifdefdirective((d: any) => d.name(spelled("A")).branches([(b: any) => b]))]) {
    raises(ValueError, () => text(spec), "has no name or condition to print");
  }
}

<a id="SVD-09"></a>
## SVD-09 · Evaluation: none (Q11); each of its kinds is refused by name, wherever it stands

In [ ]:
{
  const cases: [string, unknown][] = [
    ["macrousage", (u: any) => u.macrousage((m: any) => m.name(spelled("W")))],
    ["includedirective", (u: any) => u.includedirective((d: any) => d.path("a.svh"))],
    ["definedirective", (u: any) => u.definedirective((d: any) => d.name(spelled("W")))],
    ["undefdirective", (u: any) => u.undefdirective((d: any) => d.name(spelled("W")))],
    ["timescaledirective", (u: any) => u.timescaledirective((d: any) => d.unit("1ns").precision("1ps"))],
    ["defaultnettypedirective", (u: any) => u.defaultnettypedirective((d: any) => d.net_type("wire"))],
    ["ifdefdirective", ifdef], ["disabledtext", (u: any) => u.disabledtext((t: any) => t.text("a"))],
    ["otherdirective", (u: any) => u.otherdirective((d: any) => d.text("`resetall"))], ["comment", comment(" c")],
  ];
  for (const [tag, spec] of cases) {
    raises(NotImplementedError, () => SVV.OfAny(term(spec), { x: 1n }), `SystemVerilog Preprocessor ${tag} is not evaluated`);
  }
  raises(NotImplementedError, () => SVV.OfAny(term(SV.binary("+", (u: any) => u.macrousage((m: any) => m.name(spelled("W"))), 1n), SV.DIALECT)),
    "SystemVerilog Preprocessor macrousage is not evaluated"); // within an expression
}